In [0]:
%sql
CREATE OR REPLACE TABLE boc_silver_merge_demo
USING DELTA
AS
SELECT *
FROM boc_silver_exchange_rates;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_silver AS

WITH latest_record AS (
    SELECT
        observation_date,
        series,
        value
    FROM boc_silver_merge_demo
    ORDER BY observation_date DESC
    LIMIT 1
)

SELECT
    observation_date,
    series,
    value + 0.01 AS value
FROM latest_record

UNION ALL

SELECT
    DATE_ADD(observation_date, 1) AS observation_date,
    series,
    value + 0.02 AS value
FROM latest_record;

In [0]:
%sql
MERGE INTO boc_silver_merge_demo AS target

USING incoming_silver AS source

ON target.observation_date = source.observation_date
AND target.series = source.series

WHEN MATCHED THEN
    UPDATE SET
        target.value = source.value

WHEN NOT MATCHED THEN
    INSERT (
        observation_date,
        series,
        value
    )
    VALUES (
        source.observation_date,
        source.series,
        source.value
    );

In [0]:
%sql
SELECT *
FROM boc_silver_merge_demo
ORDER BY observation_date DESC
LIMIT 10;